# ⚙️ Paradigmas y Modelos de Difusión

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mattbarreto/ifts24-lab-pdi-2026/blob/master/009%20-%20modelos_difusion/02_Paradigmas_y_Modelos_Difusion.ipynb)

**Procesamiento Digital de Imágenes — IFTS24**
Prof. Matias Barreto — 2026

**Unidad 9 · Bloque 2 — 50 minutos**

---

## Al terminar este bloque vas a poder:

1. Comparar los tres paradigmas de generación de imágenes: GAN, VAE y difusión.
2. Identificar las ventajas y limitaciones de cada enfoque en términos de calidad y control.
3. Reconocer los componentes internos de un modelo de difusión latente (UNet, VAE, CLIP).
4. Seleccionar el modelo más apropiado según los requerimientos de un caso de uso.

---

### ✎ Para pensar

1. Las GAN tienen problemas de estabilidad en el entrenamiento. ¿Por qué es difícil mantener en equilibrio al generador y al discriminador?
2. Si una GAN genera imágenes mucho más rápido que un modelo de difusión, ¿por qué la difusión la reemplazó en muchas aplicaciones?

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **GAN** | Red Adversarial Generativa: dos redes en competencia — un generador y un discriminador — que se entrenan mutuamente hasta que el generador produce imágenes convincentes. |
| **VAE** | Autoencoder Variacional: modelo que comprime imágenes a un espacio latente y luego las reconstruye aprendiendo una distribución probabilística continua. |
| **Destilación de modelos** | Técnica que comprime el conocimiento de un modelo grande en uno más pequeño y rápido sin reentrenar desde cero con datos originales. |
| **Pipeline** | Cadena de componentes (encoder, UNet, decoder, scheduler) que se ejecutan en secuencia para generar una imagen desde un prompt. |
| **Stable Diffusion** | Familia de modelos de difusión latente que operan en espacio comprimido para ser más eficientes en memoria y tiempo. |
| **Texto condicional** | Capacidad de guiar la generación a partir de una descripción en lenguaje natural; implementada mediante el encoder CLIP o T5. |

---

## 1. Repaso: ¿Qué Hicimos Hasta Ahora en el Curso?

### Módulos Previos

**Módulo 1-2: Fundamentos**
- Aprendiste que una imagen es una matriz de números
- Trabajaste con espacios de color (RGB, escala de grises)
- Entendiste resolución, muestreo, cuantización

**Módulo 3-4: Segmentación**
- Detectaste objetos por color
- Creaste máscaras binarias
- Hiciste tu primer proyecto (detector de frutillas)

**Módulo 5: Filtros y Convolución**
- Aplicaste filtros de suavizado (Gaussiano)
- Detectaste bordes (Sobel, Laplaciano)
- Entendiste cómo funcionan las convoluciones

**Módulo 6: Deep Learning**
- Construiste CNNs con Keras
- Clasificaste imágenes con redes neuronales
- Usaste modelos pre-entrenados

### El Paradigma Tradicional

Hasta ahora, siempre trabajamos así:

```
IMAGEN EXISTENTE → ANÁLISIS/TRANSFORMACIÓN → RESULTADO
```

Ejemplos:
- Imagen con ruido → Filtro gaussiano → Imagen suavizada
- Imagen con objetos → CNN → Clasificación
- Imagen a color → Segmentación → Máscara de objetos

**Característica clave:** Siempre partimos de una imagen que ya existe.

---

## 2. El Nuevo Paradigma: Generación de Imágenes

### ¿Qué Cambia con Modelos Generativos?

```
DESCRIPCIÓN DE TEXTO → GENERACIÓN → IMAGEN NUEVA
```

O también:

```
IMAGEN + INSTRUCCIONES → MODIFICACIÓN → IMAGEN TRANSFORMADA
```

### Comparación Visual

| Enfoque | Input | Proceso | Output |
|---------|-------|---------|--------|
| **Tradicional** | Imagen existente | Transformación matemática | Imagen modificada |
| **Generativo** | Texto o ruido aleatorio | Modelo de difusión | Imagen creada desde cero |

### Ejemplos de Aplicaciones

**Text-to-Image:**
- Input: "Un gato naranja durmiendo en un sillón azul"
- Output: Imagen fotorealista de exactamente eso

**Inpainting:**
- Input: Foto con un objeto no deseado + máscara
- Output: Foto con ese objeto eliminado y el fondo rellenado coherentemente

**Super-Resolution:**
- Input: Imagen de 128x128 píxeles
- Output: Imagen de 512x512 con detalles realistas agregados

**Image-to-Image:**
- Input: Foto de día + "convert to sunset"
- Output: La misma escena pero en atardecer

---

## 3. ¿Qué son los Modelos de Difusión?

### Concepto Fundamental

Los modelos de difusión aprenden a **revertir un proceso de degradación gradual**.

### El Proceso en Dos Fases

**Fase 1: Forward Process (Agregar Ruido)**

```
Imagen Limpia → +ruido → +ruido → +ruido → ... → Ruido Puro
  (t=0)        (t=1)    (t=2)    (t=3)         (t=1000)
```

Este proceso es determinístico: sabemos exactamente cómo agregar ruido.

**Fase 2: Reverse Process (Eliminar Ruido)**

```
Ruido Puro → -ruido → -ruido → -ruido → ... → Imagen Generada
  (t=1000)   (t=999)  (t=998)  (t=997)        (t=0)
```

Este proceso es lo que el modelo **aprende** a hacer.

### Analogía

Imaginá que tenés una foto y le vas tirando pintura encima gradualmente hasta que no se ve nada. El modelo aprende a "limpiar" esa pintura paso a paso hasta recuperar una imagen coherente.

La magia está en que puede generar imágenes nuevas (que nunca existieron) partiendo de ruido aleatorio y "limpiándolo" siguiendo las instrucciones de un texto.

---

## 4. Diferencias con Métodos Tradicionales

### Filtro Gaussiano (Módulo 5) vs Modelo de Difusión

**Filtro Gaussiano (Tradicional):**
- Aplica una operación matemática fija
- Promedia valores de píxeles vecinos
- Suaviza el ruido pero pierde detalles
- No "entiende" qué hay en la imagen
- Muy rápido (milisegundos)

**Modelo de Difusión (Generativo):**
- Usa una red neuronal entrenada con millones de imágenes
- "Entiende" estructuras (caras, objetos, texturas)
- Puede reconstruir detalles perdidos
- Proceso iterativo (múltiples pasos)
- Más lento (segundos a minutos)

### CNN Clasificadora (Módulo 6) vs Modelo de Difusión

**CNN Clasificadora:**
- Input: Imagen → Output: Etiqueta
- Ejemplo: Imagen de gato → "gato" (90% confianza)
- Una sola pasada forward
- Discriminativa (distingue categorías)

**Modelo de Difusión:**
- Input: Texto/Ruido → Output: Imagen
- Ejemplo: "gato" → Imagen de gato
- Múltiples pasadas iterativas
- Generativa (crea contenido nuevo)

### ¿Cuándo Usar Cada Una?

| Tarea | Método Tradicional | Modelo de Difusión |
|-------|-------------------|-------------------|
| Suavizar ruido simple | Filtro Gaussiano ✓ | Overkill |
| Detectar bordes | Sobel/Canny ✓ | No aplica |
| Clasificar imagen | CNN ✓ | No aplica |
| Eliminar ruido complejo | Limitado | Difusión ✓ |
| Generar imagen nueva | No puede | Difusión ✓ |
| Rellenar regiones faltantes | Inpainting básico | Difusión ✓ |
| Aumentar resolución con detalles | Interpolación básica | Difusión ✓ |

---

## 5. Evolución de Modelos de Difusión (2022-2025)

### Línea de Tiempo

**2022: Stable Diffusion v1.5**
- Primer modelo de difusión open source popular
- 512x512 píxeles
- 20-50 pasos de inferencia
- ~10-20 segundos por imagen en GPU media

**2023: Stable Diffusion XL (SDXL)**
- Mejor calidad, más detalles
- 1024x1024 píxeles nativos
- Mejor comprensión de prompts complejos
- Más lento pero mucho mejor resultado

**2023: Turbo Models**
- SDXL-Turbo y SD-Turbo
- Destilación de conocimiento
- Reducción de 50 pasos a **1 solo paso**
- Velocidad x50 sin perder mucha calidad

**2024: FLUX.1**
- Nueva arquitectura
- Calidad superior a SDXL
- FLUX.1-schnell: 1-4 pasos, ultra-rápido
- Mejor manejo de texto en imágenes

**2024: SDXS (Stable Diffusion XS)**
- Modelo ultra-comprimido
- Diseñado específicamente para CPU
- 100 FPS en GPU, ~1 segundo en CPU
- Sacrifica un poco de calidad por velocidad

**2024-2025: LCM-LoRA**
- Módulo acelerador universal
- Se agrega a cualquier modelo SD
- Reduce pasos de 50 a 4-8
- Mantiene compatibilidad con checkpoints existentes

---

## 6. Modelos Principales en 2025

### Comparación Rápida

| Modelo | Velocidad | Calidad | VRAM | Mejor Para |
|--------|-----------|---------|------|------------|
| **FLUX.1-schnell** | ✦ | ✦✦✦✦✦ | 8+ GB | Máxima calidad, GPU potente |
| **SDXL-Turbo** | ✦ | ✦✦✦✦✦ | 6-8 GB | Balance perfecto |
| **SD-Turbo** | ✦ | ✦✦✦✦ | 4 GB | GPU modesta |
| **SDXS-512** | ✦ | ✦✦✦✦ | 2 GB | CPU viable |
| **LCM-LoRA** | ✦ | ✦✦✦✦ | Variable | Acelerar modelos existentes |

### ¿Cuál Usar?

**Preguntate:**

1. ¿Dónde voy a trabajar?
   - Google Colab → FLUX o SDXL-Turbo
   - PC con GPU → Según VRAM
   - PC sin GPU → SDXS-512

2. ¿Qué GPU tengo? (si tengo)
   - 8+ GB → FLUX.1-schnell
   - 4-8 GB → SDXL-Turbo
   - <4 GB → SD-Turbo con optimizaciones

3. ¿Qué es más importante?
   - Calidad máxima → FLUX
   - Balance → SDXL-Turbo
   - Velocidad en CPU → SDXS

Ver tabla detallada en: `002_Tabla_Comparativa_Modelos.md`

---

## 7. Demo Práctica: Tu Primera Generación

Vamos a hacer una demostración simple usando el modelo más compatible (funciona en la mayoría de los hardwares).

### Importante

Esta es una demo introductoria. Los notebooks específicos de cada modelo tienen explicaciones más detalladas y optimizaciones.

In [ ]:
# Paso 1: Instalar librerías necesarias
# Si estás en Google Colab, esta celda instalará todo lo necesario
# Si estás en local, asegurate de tener el entorno virtual activado

!pip install diffusers transformers accelerate torch -q

In [ ]:
# Paso 2: Importar librerías

import torch
from diffusers import DiffusionPipeline
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente")
print(f"Versión de PyTorch: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")

In [ ]:
# Paso 3: Detectar hardware disponible

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detectada: {gpu_name}")
    print(f"VRAM disponible: {gpu_memory:.1f} GB")

    # Seleccionar modelo según VRAM
    if gpu_memory >= 8:
        modelo_recomendado = "stabilityai/sdxl-turbo"
        print("\nUsando SDXL-Turbo (alta calidad, 1024x1024)")
    else:
        modelo_recomendado = "stabilityai/sd-turbo"
        print("\nUsando SD-Turbo (optimizado, 512x512)")
else:
    device = "cpu"
    modelo_recomendado = "stabilityai/sd-turbo"
    print("No se detectó GPU, usando CPU")
    print("Advertencia: La generación va a ser lenta en CPU")
    print("Recomendación: Usar Google Colab con GPU o el notebook SDXS para CPU")

In [ ]:
# Paso 4: Cargar el modelo
# Esta celda puede tardar 2-5 minutos la primera vez (descarga el modelo)

print(f"Cargando modelo: {modelo_recomendado}")
print("Esto puede tardar unos minutos la primera vez...\n")

pipe = DiffusionPipeline.from_pretrained(
    modelo_recomendado,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
)

pipe = pipe.to(device)

print("Modelo cargado exitosamente")

In [ ]:
# Paso 5: Generar tu primera imagen

# Escribí acá tu descripción (en inglés funciona mejor)
prompt = "a beautiful sunset over mountains, vibrant colors, highly detailed"

print(f"Generando imagen: '{prompt}'")
print("Procesando...\n")

# Generar
image = pipe(
    prompt=prompt,
    num_inference_steps=1,  # Turbo models usan 1 solo paso
    guidance_scale=0.0      # Turbo models no usan guidance
).images[0]

# Mostrar
display(image)
print("\nImagen generada exitosamente")

### ¿Qué Acabó de Pasar?

1. **Cargaste un modelo** entrenado con millones de imágenes
2. **Escribiste un prompt** describiendo lo que querés
3. El modelo **partió de ruido aleatorio**
4. En **1 solo paso** (gracias a Turbo), eliminó el ruido siguiendo tu descripción
5. **Generó una imagen nueva** que nunca existió antes

### Experimentá

Probá cambiar el prompt en la celda anterior. Algunas ideas:

- `"a cute orange cat sleeping on a blue couch"`
- `"a futuristic city at night with neon lights"`
- `"a delicious pizza with cheese and tomatoes"`
- `"a peaceful forest with sunlight through trees"`

In [ ]:
# Ejercicio: Generá 4 imágenes con diferentes prompts

prompts = [
    "a red apple on a wooden table",
    "a modern house with large windows",
    "a colorful parrot in the jungle",
    "a steaming cup of coffee on a desk"
]

print("Generando 4 imágenes...\n")

for i, p in enumerate(prompts, 1):
    print(f"{i}. {p}")
    img = pipe(prompt=p, num_inference_steps=1, guidance_scale=0.0).images[0]
    display(img)
    print()

### ✎ Para pensar

1. ¿Por qué Stable Diffusion opera en 'espacio latente' en lugar de directamente sobre los píxeles de la imagen?
2. ¿Qué rol cumple CLIP en el proceso de generación? ¿Qué pasaría si el modelo no tuviera esa componente?

---

## 8. Conceptos Clave para Recordar

### Diferencia Fundamental

**Métodos Tradicionales (Módulos 1-6):**
- Transforman imágenes existentes
- Operaciones matemáticas determinísticas
- Rápidos pero limitados
- No "entienden" contenido semántico

**Modelos de Difusión:**
- Generan o reconstruyen imágenes
- Proceso iterativo estocástico
- Más lentos pero mucho más capaces
- "Entienden" contenido y contexto

### Proceso de Difusión en Resumen

1. **Forward:** Imagen limpia → Ruido puro (proceso conocido)
2. **Reverse:** Ruido puro → Imagen coherente (proceso aprendido)
3. **Condicionamiento:** El texto guía el proceso reverse

### Evolución Clave

- 2022: SD v1.5 (50 pasos, lento)
- 2023: SDXL (mejor calidad)
- 2023: Turbo models (1 paso, rápido)
- 2024: FLUX (máxima calidad)
- 2024: SDXS (viable en CPU)

### Hardware y Modelos

- GPU 8+ GB → FLUX.1-schnell
- GPU 4-8 GB → SDXL-Turbo
- GPU <4 GB → SD-Turbo
- Sin GPU → SDXS-512

---

## 9. Conexión con Módulos Previos

### Módulo 5: Filtros y Convolución

**Lo que aprendiste:**
- Las CNNs usan convoluciones para extraer características
- Los filtros detectan patrones (bordes, texturas)

**Conexión con Difusión:**
- Los modelos de difusión también usan CNNs internamente (U-Net)
- Pero en vez de clasificar, reconstruyen progresivamente
- Las convoluciones aprenden a predecir qué ruido eliminar

### Módulo 6: Deep Learning

**Lo que aprendiste:**
- Redes discriminativas: Imagen → Etiqueta
- Transfer learning: Usar modelos pre-entrenados

**Conexión con Difusión:**
- Redes generativas: Texto/Ruido → Imagen
- También usamos modelos pre-entrenados (no entrenamos desde cero)
- Pero el objetivo es crear, no clasificar

### Módulo 3: Segmentación

**Lo que aprendiste:**
- Crear máscaras binarias para separar objetos

**Conexión con Difusión:**
- Las máscaras se usan para inpainting
- Combinás segmentación tradicional con generación IA
- Pipeline: Detectar objeto → Crear máscara → Inpainting con difusión

---

## 10. Limitaciones y Consideraciones

### Cuándo NO Usar Modelos de Difusión

**No usar si:**
- Necesitás velocidad en tiempo real (usa filtros tradicionales)
- Querés resultados 100% determinísticos
- No tenés GPU y el modelo no está optimizado para CPU
- La tarea es simple (suavizar, rotar, cambiar brillo)

### Limitaciones Técnicas

**Hardware:**
- Requieren GPU para ser prácticos (excepto SDXS)
- Consumen mucha VRAM
- Los modelos ocupan varios GB de almacenamiento

**Resultados:**
- No siempre generan exactamente lo que pedís
- Pueden incluir artefactos o inconsistencias
- Tienen dificultades con texto dentro de imágenes
- Pueden generar contenido sesgado (según datos de entrenamiento)

**Iteración:**
- A veces necesitás generar varias veces hasta obtener buen resultado
- Los prompts requieren práctica para obtener lo que querés

### Consideraciones Éticas

**Importantes:**
- No generés contenido para desinformar o engañar
- Respetá derechos de autor y marcas
- No generés contenido dañino o ilegal
- Sé transparente cuando uses IA para generar contenido
- Revisá las licencias de cada modelo antes de uso comercial

---

## 11. Próximos Pasos

### En Este Módulo

**Carpeta 02: Generación de Imágenes**
- Elegí el notebook según tu hardware
- Aprendé a optimizar uso de memoria
- Experimentá con diferentes modelos

**Carpeta 03: Aplicaciones Prácticas**
- Inpainting: Eliminar objetos de fotos
- Super-resolution: Mejorar calidad de imágenes
- Image-to-image: Transformar características
- Denoising: Eliminar ruido profesionalmente

**Carpeta 04: Integración con el Curso**
- Combinar filtros tradicionales con IA
- Usar segmentación + generación
- Pipelines completos de procesamiento

**Carpeta 05: Deployment**
- Crear aplicación web con Streamlit
- Dockerizar tu proyecto
- Publicar en Hugging Face Spaces

### Recomendación de Ruta

1. Terminá este notebook
2. Revisá la tabla comparativa: `002_Tabla_Comparativa_Modelos.md`
3. Elegí el notebook de generación según tu hardware:
   - GPU 8+ GB: `010_Text_to_Image_Flux_Schnell.ipynb`
   - GPU 4-8 GB: `011_Text_to_Image_SDXL_Turbo.ipynb`
   - Sin GPU: `012_Text_to_Image_SDXS_CPU.ipynb`
4. Seguí con aplicaciones prácticas
5. Integración y proyecto final

---

## 12. Recursos Adicionales

### Documentación Oficial

- **Hugging Face Diffusers:** https://huggingface.co/docs/diffusers/
- **FLUX:** https://huggingface.co/black-forest-labs/FLUX.1-schnell
- **SDXL:** https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0

### Papers Importantes

- Denoising Diffusion Probabilistic Models (Ho et al., 2020)
- High-Resolution Image Synthesis with Latent Diffusion Models (Rombach et al., 2022)
- SDXL (Podell et al., 2023)
- SDXS (IDKiro et al., 2024)

### En Este Módulo

- `06_Material_Apoyo/Glosario_Tecnico.md` - Diccionario de términos
- `06_Material_Apoyo/Troubleshooting_Errores_Comunes.md` - Soluciones
- `06_Material_Apoyo/Guia_Prompts_Efectivos.md` - Escribir mejores prompts
- `06_Material_Apoyo/Recursos_Adicionales.md` - Links útiles

---

## Resumen Final

### Lo que Aprendiste Hoy

- ✓ Diferencia entre procesamiento tradicional y generativo
- ✓ Cómo funcionan los modelos de difusión (forward + reverse)
- ✓ Evolución 2022-2025 de los modelos
- ✓ Qué modelo elegir según tu hardware
- ✓ Tu primera generación de imagen con IA

### Conceptos Clave

1. Los modelos de difusión **generan** imágenes, no solo las transforman
2. Aprenden a **revertir** un proceso de agregado de ruido
3. Los modelos Turbo reducen de 50 pasos a **1 paso**
4. Diferentes modelos para diferentes **hardwares**
5. Se **complementan** con métodos tradicionales, no los reemplazan

### Siguiente Notebook

Según tu hardware, andá a:
- `02_Generacion_Imagenes/010_Text_to_Image_Flux_Schnell.ipynb` (GPU potente)
- `02_Generacion_Imagenes/011_Text_to_Image_SDXL_Turbo.ipynb` (GPU media)
- `02_Generacion_Imagenes/012_Text_to_Image_SDXS_CPU.ipynb` (sin GPU)

---


## ⛰️ Lo que construimos

| Paradigma | Fortaleza principal | Limitación clave |
|---|---|---|
| GAN | Velocidad de generación | Inestabilidad en entrenamiento |
| VAE | Interpolación suave en espacio latente | Imágenes más borrosas |
| Difusión | Calidad y diversidad superior | Lentitud (muchos pasos) |
| Difusión latente | Eficiencia + calidad | Complejidad de componentes |

**Próximo cuaderno:** `03_Aplicaciones_Practicas_Difusion` — usamos inpainting, image-to-image y outpainting en la práctica.